## Libraries

In [1]:
import os
from dotenv import load_dotenv
from supabase import create_client
import pandas as pd
import numpy as np
from IPython.display import display
import math
import json
from pathlib import Path

## Connect to Supabase and Test the Connection

In [2]:
# Retrieve Supabase project URL and API key from environment variables

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

print("Supabase URL found:", bool(SUPABASE_URL))
print("Supabase key found:", bool(SUPABASE_KEY))

Supabase URL found: True
Supabase key found: True


In [3]:
# Check whether the Supabase URL and API key are available.
# Stop execution if either environment variable is missing.

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("Check the variable names in your .env file.")

# Create a Supabase client using the project URL and API key
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Run a simple read-only query on the customers table
response = (
    supabase.table("customers")
    .select("customer_id")
    .limit(1)
    .execute()
)

# Display the connection status and number of records returned
print("Supabase connection successful!")
print("Test query returned:", len(response.data), "row(s)")

Supabase connection successful!
Test query returned: 1 row(s)


## Load the Full Supabase Table

 1. customers
 2. merchants
 3. devices
 4. transactions
 5. payment_outcomes
 6. labels

In [4]:
def load_full_table(table_name, page_size=1000):
    """
    It downloads the entire specified table in batches of 1000 records, combines the batches 
    and returns the complete data as a pandas DataFrame
    
    """
    # Create an empty list to store records fetched from Supabase
    all_records = []
    # Set the starting row index for pagination
    start = 0

    # Continue fetching records until all rows have been retrieved
    while True:
        # Fetch a batch of records from the specified table
        response = (
            supabase.table(table_name)
            .select("*")
            .range(start, start + page_size - 1)
            .execute()
        )

        records = response.data
        all_records.extend(records)

        if len(records) < page_size:
            break

        start += page_size

    return pd.DataFrame(all_records)

In [5]:
# List of Required Supabase Tables

sentinel_data_tables = [
    "customers",
    "merchants",
    "devices",
    "transactions",
    "payment_outcomes",
    "labels"
]

In [6]:
# Load tables and display sample records

 # Store each table's full data 
sentinel_data = {}

# Store row and column counts for each table
summary = []

# Load all records from the current table
for table_name in sentinel_data_tables:
    df = load_full_table(table_name)
    sentinel_data[table_name] = df

    # Saves the row and column counts for the current table in the summary list
    summary.append({
        "Table Name": table_name,
        "Total Rows": len(df),
        "Total Columns": len(df.columns)
    })

    print(f"\n{'=' * 140}")
    print(f"Table: {table_name}")
    print(f"Total rows and columns: ({len(df):,} | {len(df.columns)})")
    print("Sample records:")

    display(df.head(5))


Table: customers
Total rows and columns: (2,500 | 6)
Sample records:


,customer_id,created_at,customer_age,home_city,region,is_synthetic
0,C_7df0dbcdc76f4e15,2022-06-13T18:30:00+00:00,78,Hyderabad,south,True
1,C_0e5ad51fc4f69677,2023-01-15T18:30:00+00:00,65,Kochi,south,True
2,C_5bc1f3093ad18d0b,2022-12-29T18:30:00+00:00,20,Guwahati,northeast,True
3,C_ac88640ab1035671,2023-12-11T18:30:00+00:00,61,Jaipur,north,True
4,C_8dfbff5e9461db4d,2024-06-30T18:30:00+00:00,62,Patna,east,True



Table: merchants
Total rows and columns: (200 | 5)
Sample records:


,merchant_id,synthetic_name,merchant_category,merchant_location,is_synthetic
0,M_3d94d2f3806d948e,Synthetic Merchant 000000,grocery,"{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}",True
1,M_36408902719733e3,Synthetic Merchant 000001,food,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}",True
2,M_70031a4f1ff10754,Synthetic Merchant 000002,education,"{'lat': 28.61, 'lon': 77.21, 'city': 'Delhi'}",True
3,M_1803b1b25f57f19f,Synthetic Merchant 000003,travel,"{'lat': 18.52, 'lon': 73.86, 'city': 'Pune'}",True
4,M_3d0c08f9e832bdc9,Synthetic Merchant 000004,electronics,"{'lat': 17.39, 'lon': 78.49, 'city': 'Hyderabad'}",True



Table: devices
Total rows and columns: (10,120 | 3)
Sample records:


...

,transaction_id,customer_id,merchant_id,device_id,amount_paise,transaction_amount,payment_channel,merchant_category,transaction_time,customer_age,customer_location,merchant_location,account_age,previous_transactions,average_transaction_value,failed_transactions,split,is_synthetic
0,TX_0000000501,C_7c4a55457f5223f2,M_290cf69ca3b60fe3,D_d04661e827af262c,813105,8131.05,card,other,2024-10-05T08:47:07.534+00:00,71,"{'lat': 28.61, 'lon': 77.21, 'city': 'Delhi'}","{'lat': 25.61, 'lon': 85.14, 'city': 'Patna'}",57,2,7681.92,0,warmup,True
1,TX_0000000502,C_7c4a55457f5223f2,M_290cf69ca3b60fe3,D_d04661e827af262c,286321,2863.21,card,other,2024-10-05T08:48:00.607+00:00,71,"{'lat': 28.61, 'lon': 77.21, 'city': 'Delhi'}","{'lat': 25.61, 'lon': 85.14, 'city': 'Patna'}",57,3,7831.63,0,warmup,True
2,TX_0000000503,C_7c4a55457f5223f2,M_290cf69ca3b60fe3,D_d04661e827af262c,157400,1574.00,card,other,2024-10-05T08:49:10.473+00:00,71,"{'lat': 28.61, 'lon': 77.21, 'city': 'Delhi'}","{'lat': 25.61, 'lon': 85.14, 'city': 'Patna'}",57,4,6589.53,0,warmup,True
3,TX_0000000504,C_7c4a55457f5223f2,M_290cf69ca3b60fe3,D_d04661e827af262c,2148244,21482.44,card,other,2024-10-05T08:51:19.349+00:00,71,"{'lat': 28.61, 'lon': 77.21, 'city': 'Delhi'}","{'lat': 25.61, 'lon': 85.14, 'city': 'Patna'}",57,5,5586.42,0,warmup,True
4,TX_0000000505,C_7c4a55457f5223f2,M_290cf69ca3b60fe3,D_d04661e827af262c,403784,4037.84,card,other,2024-10-05T08:51:59.97+00:00,71,"{'lat': 28.61, 'lon': 77.21, 'city': 'Delhi'}","{'lat': 25.61, 'lon': 85.14, 'city': 'Patna'}",57,6,8235.76,0,warmup,True



Table: payment_outcomes
Total rows and columns: (110,000 | 5)
Sample records:


,transaction_id,outcome_time,status,failure_code,is_synthetic
0,TX_0000000501,2024-10-05T08:47:08.762+00:00,SUCCESS,NaN,True
1,TX_0000000502,2024-10-05T08:48:01.632+00:00,SUCCESS,NaN,True
2,TX_0000000503,2024-10-05T08:49:11.629+00:00,SUCCESS,NaN,True
3,TX_0000000504,2024-10-05T08:51:20.59+00:00,SUCCESS,NaN,True
4,TX_0000000505,2024-10-05T08:52:00.35+00:00,SUCCESS,NaN,True



Table: labels
Total rows and columns: (110,000 | 7)
Sample records:


,transaction_id,fraud_label,label_source,label_available_at,split,eligible_at_next_split_start,is_synthetic
0,TX_0000001501,0,mature_no_fraud,2024-12-13T15:43:30.771+00:00,warmup,True,True
1,TX_0000000501,0,mature_no_fraud,2024-12-04T08:47:07.534+00:00,warmup,True,True
2,TX_0000000001,0,mature_no_fraud,2024-11-29T18:35:15.602+00:00,warmup,True,True
3,TX_0000000002,0,mature_no_fraud,2024-11-29T18:36:10.733+00:00,warmup,True,True
4,TX_0000000502,0,mature_no_fraud,2024-12-04T08:48:00.607+00:00,warmup,True,True


In [7]:
# SENTINEL data summary
print("---- Summary of all tables ----")
display(pd.DataFrame(summary))

---- Summary of all tables ----


,Table Name,Total Rows,Total Columns
0,customers,2500,6
1,merchants,200,5
2,devices,10120,3
3,transactions,110000,18
4,payment_outcomes,110000,5
5,labels,110000,7


## Reusable Data Quality Functions

In [8]:
# Display table dimensions, column names and data types
def inspect_table(df, table_name):
    print(f"\n{'=' * 70}")
    print(f"Data Quality Report: {table_name}")
    print(f"{'=' * 70}")

    print(f"Total rows: {df.shape[0]:,}")
    print(f"Total columns: {df.shape[1]}")

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nColumn data types:")
    display(df.dtypes.rename("Data Type").to_frame())

In [9]:
# Validate actual data types against expected data types
def check_data_types(df, expected_types):
    print(f"Data Type Validation:")

    type_report = []

    for column in df.columns:
        actual_type = str(df[column].dtype)
        expected = expected_types.get(column, "Not specified")

        if column not in expected_types:
            status = "Review manually"
        elif actual_type == expected:
            status = "OK"
        else:
            status = "Review"

        type_report.append({
            "Column": column,
            "Actual Type": actual_type,
            "Expected Type": expected,
            "Status": status
        })

    type_report = pd.DataFrame(type_report)
    display(type_report)

    return type_report

In [10]:
# Calculate null counts and percentages per column
def check_nulls(df):
    print(f"\nMissing Value Report:")

    null_count = df.isnull().sum()

    null_report = pd.DataFrame({
        "Null Count": null_count,
        "Null Percentage (%)": (
            null_count / len(df) * 100 if len(df) > 0
            else 0
        ).round(2)
    })

    display(null_report)
    return null_report

In [ ]:
# Check Duplicate IDs and Completely Duplicated Rows

def check_duplicates(df, id_columns=None):
    print("\nDuplicate Analysis:")

    # Create a temporary copy for duplicate checking.
    # Convert dictionaries and lists into comparable representations.
    df_check = df.copy()

    def make_hashable(value):
        if isinstance(value, dict):
            return tuple(sorted(
                (key, make_hashable(val))
                for key, val in value.items()
            ))
        elif isinstance(value, (list, tuple)):
            return tuple(make_hashable(item) for item in value)
        else:
            return value

    # Convert dictionary or list values without modifying the original data.
    for column in df_check.columns:
        if df_check[column].map(
            lambda value: isinstance(value, (dict, list, tuple))
        ).any():
            df_check[column] = df_check[column].map(make_hashable)

    # Count completely duplicated rows.
    duplicate_rows = int(df_check.duplicated().sum())
    print(f"Completely duplicated rows: {duplicate_rows:,}")

    # Default to no ID checks if columns are not provided.
    if id_columns is None:
        id_columns = []

    # Check duplicates for each specified ID or group of columns.
    for column in id_columns:
        columns_to_check = [column] if isinstance(column, str) else column

        missing_columns = [
            col for col in columns_to_check if col not in df.columns
        ]

        if missing_columns:
            print(
                f"Cannot check {columns_to_check}: "
                f"column(s) not found."
            )
            continue

        # IDs should be scalar values, so check them directly.
        duplicate_ids = int(
            df.duplicated(subset=columns_to_check).sum()
        )

        print(
            f"Duplicate ID values in {columns_to_check}: "
            f"{duplicate_ids:,}"
        )

    return duplicate_rows


In [12]:
# Identify potential outliers in specified numeric columns
def check_outliers(df, columns):
    print(f"\nOutlier Analysis:")

    outlier_reports = []

    for column in columns:
        if column not in df.columns:
            print(f"Column '{column}' not found. Skipping.")
            continue

        # Convert values to numeric for statistical analysis.
        values = pd.to_numeric(df[column], errors="coerce").dropna()

        if values.empty:
            print(f"Column '{column}' has no valid numeric values.")
            continue

        # Calculate quartiles and the interquartile range (IQR).
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        # Define the lower and upper IQR outlier boundaries.
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Count values outside the calculated boundaries.
        outlier_mask = (values < lower_bound) | (values > upper_bound)
        outlier_count = int(outlier_mask.sum())

        outlier_reports.append({
            "Column": column,
            "Valid Values": len(values),
            "Minimum": values.min(),
            "Q1": q1,
            "Q3": q3,
            "Maximum": values.max(),
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Outlier Count": outlier_count,
            "Outlier Percentage (%)": round(
                outlier_count / len(values) * 100, 2
            )
        })

    outlier_report = pd.DataFrame(outlier_reports)
    if not outlier_report.empty:
        display(outlier_report)

    # Skip outlier analysis when no columns are provided.
    else:
        print("Outlier analysis skipped: no suitable numeric columns.")

    return outlier_report

In [ ]:
# Check whether a split column exists and count its values
def check_splits(df, table_name):
    print(f"\nSplit Analysis:")

    if "split" not in df.columns:
        print(
            f"The '{table_name}' table does not contain a 'split' "
            "column. Split analysis is not applicable."
        )
        return None

    # Count every split value, including missing or unexpected values.
    split_counts = (
        df["split"]
        .value_counts(dropna=False)
        .rename_axis("Split")
        .reset_index(name="Row Count")
    )

    display(split_counts)

    # Check for expected splits that may be absent.
    expected_splits = ["warmup", "train", "validation", "holdout"]
    existing_splits = set(df["split"].dropna().unique())

    missing_splits = [
        split for split in expected_splits
        if split not in existing_splits
    ]

    if missing_splits:
        print("Expected splits not found:", missing_splits)
    else:
        print("All four expected splits are present.")

    print(f"Total rows counted: {len(df):,}")

    return split_counts

In [ ]:
# Run all applicable data quality checks
def run_data_quality_checks(
    df,
    table_name,
    expected_types,
    id_columns=None,
    outlier_columns=None
):

    # Run the six data quality checks.
    inspect_table(df, table_name)
    type_report = check_data_types(df, expected_types)
    null_report = check_nulls(df)
    duplicate_count = check_duplicates(df, id_columns)
    outlier_report = check_outliers(df, outlier_columns or [])
    split_report = check_splits(df, table_name)

    # Return results so they can be reused or exported later
    return {
        "type_report": type_report,
        "null_report": null_report,
        "duplicate_rows": duplicate_count,
        "outlier_report": outlier_report,
        "split_report": split_report
    }

## Data quality report of customers table

In [15]:
# Define expected data types for the customers table.
customers_expected_types = {
    "customer_id": "str",
    "created_at": "datetime64[ns, UTC]",
    "customer_age": "int64",
    "home_city": "str",
    "region": "str",
    "is_synthetic": "bool"
}

# Run all data quality checks on the complete customers dataset.
customers_quality = run_data_quality_checks(
    df=sentinel_data["customers"],
    table_name="customers",
    expected_types=customers_expected_types,
    id_columns=["customer_id"],
    outlier_columns=["customer_age"]
)



Data Quality Report: customers
Total rows: 2,500
Total columns: 6

Column names:
['customer_id', 'created_at', 'customer_age', 'home_city', 'region', 'is_synthetic']

Column data types:


,Data Type
customer_id,str
created_at,str
customer_age,int64
home_city,str
region,str
is_synthetic,bool


Data Type Validation:


,Column,Actual Type,Expected Type,Status
0,customer_id,str,str,OK
1,created_at,str,"datetime64[ns, UTC]",Review
2,customer_age,int64,int64,OK
3,home_city,str,str,OK
4,region,str,str,OK
5,is_synthetic,bool,bool,OK



Missing Value Report:


,Null Count,Null Percentage (%)
customer_id,0,0.0
created_at,0,0.0
customer_age,0,0.0
home_city,0,0.0
region,0,0.0
is_synthetic,0,0.0



Duplicate Analysis:
Completely duplicated rows: 0
Duplicate ID values in ['customer_id']: 0

Outlier Analysis:


,Column,Valid Values,Minimum,Q1,Q3,Maximum,Lower Bound,Upper Bound,Outlier Count,Outlier Percentage (%)
0,customer_age,2500,18,34.0,65.0,80,-12.5,111.5,0,0.0



Split Analysis:
The 'customers' table does not contain a 'split' column. Split analysis is not applicable.


## Data quality report of merchants table

In [16]:
# Define expected data types for the merchants table.
merchants_expected_types = {
    "merchant_id": "str",
    "synthetic_name": "str",
    "merchant_category": "str",
    "merchant_location": "object",
    "is_synthetic": "bool"
}

# Run all data quality checks on the complete merchants dataset
merchants_quality = run_data_quality_checks(
    df=sentinel_data["merchants"],
    table_name="merchants",
    expected_types=merchants_expected_types,
    id_columns=["merchant_id"],
    outlier_columns=[]
)


Data Quality Report: merchants
Total rows: 200
Total columns: 5

Column names:
['merchant_id', 'synthetic_name', 'merchant_category', 'merchant_location', 'is_synthetic']

Column data types:


,Data Type
merchant_id,str
synthetic_name,str
merchant_category,str
merchant_location,object
is_synthetic,bool


Data Type Validation:


,Column,Actual Type,Expected Type,Status
0,merchant_id,str,str,OK
1,synthetic_name,str,str,OK
2,merchant_category,str,str,OK
3,merchant_location,object,object,OK
4,is_synthetic,bool,bool,OK



Missing Value Report:


,Null Count,Null Percentage (%)
merchant_id,0,0.0
synthetic_name,0,0.0
merchant_category,0,0.0
merchant_location,0,0.0
is_synthetic,0,0.0



Duplicate Analysis:
Completely duplicated rows: 0
Duplicate ID values in ['merchant_id']: 0

Outlier Analysis:
Outlier analysis skipped: no suitable numeric columns.

Split Analysis:
The 'merchants' table does not contain a 'split' column. Split analysis is not applicable.


## Data quality report of device table

In [ ]:
# Define expected data types for the devices table
devices_expected_types = {
    "device_id": "str",
    "device_kind": "str",
    "is_synthetic": "bool"
}

# Run all data quality checks on the complete device dataset
device_quality = run_data_quality_checks(
    df=sentinel_data["devices"],
    table_name="devices",
    expected_types=devices_expected_types,
    id_columns=["device_id"],
    outlier_columns=[]
)


Data Quality Report: devices
Total rows: 10,120
Total columns: 3

Column names:
['device_id', 'device_kind', 'is_synthetic']

Column data types:


,Data Type
device_id,str
device_kind,str
is_synthetic,bool


Data Type Validation:


,Column,Actual Type,Expected Type,Status
0,device_id,str,str,OK
1,device_kind,str,str,OK
2,is_synthetic,bool,bool,OK



Missing Value Report:


,Null Count,Null Percentage (%)
device_id,0,0.0
device_kind,0,0.0
is_synthetic,0,0.0



Duplicate Analysis:
Completely duplicated rows: 0
Duplicate ID values in ['device_id']: 0

Outlier Analysis:
Outlier analysis skipped: no suitable numeric columns.

Split Analysis:
The 'devices' table does not contain a 'split' column. Split analysis is not applicable.


## Data quality report of transactions table

In [ ]:
# Define expected data types for the transactions table
transactions_expected_types = {
    "transaction_id": "str",
    "customer_id": "str",
    "merchant_id": "str",
    "device_id": "str",
    "amount_paise": "float64",
    "transaction_amount": "float64",
    "payment_channel": "str",
    "merchant_category": "str",
    "transaction_time": "datetime64[ns, UTC]",
    "customer_age": "int64",
    "customer_location": "object",
    "merchant_location": "object",
    "account_age": "int64",
    "previous_transactions": "int64",
    "average_transaction_value": "float64",
    "failed_transactions": "int64",
    "split": "str",
    "is_synthetic": "bool"
}

# Run all data quality checks on the complete transactions dataset
transactions_quality = run_data_quality_checks(
    df=sentinel_data["transactions"],
    table_name="transactions",
    expected_types=transactions_expected_types,
    id_columns=["transaction_id"],
    outlier_columns=[
        "amount_paise",
        "transaction_amount",
        "customer_age",
        "account_age",
        "previous_transactions",
        "average_transaction_value",
        "failed_transactions"
    ]
)


Data Quality Report: transactions
Total rows: 110,000
Total columns: 18

Column names:
['transaction_id', 'customer_id', 'merchant_id', 'device_id', 'amount_paise', 'transaction_amount', 'payment_channel', 'merchant_category', 'transaction_time', 'customer_age', 'customer_location', 'merchant_location', 'account_age', 'previous_transactions', 'average_transaction_value', 'failed_transactions', 'split', 'is_synthetic']

Column data types:


,Data Type
transaction_id,str
customer_id,str
merchant_id,str
device_id,str
amount_paise,int64
transaction_amount,float64
payment_channel,str
merchant_category,str
transaction_time,str
customer_age,int64


Data Type Validation:


,Column,Actual Type,Expected Type,Status
0,transaction_id,str,str,OK
1,customer_id,str,str,OK
2,merchant_id,str,str,OK
3,device_id,str,str,OK
4,amount_paise,int64,float64,Review
5,transaction_amount,float64,float64,OK
6,payment_channel,str,str,OK
7,merchant_category,str,str,OK
8,transaction_time,str,"datetime64[ns, UTC]",Review
9,customer_age,int64,int64,OK



Missing Value Report:


...


Duplicate Analysis:
Completely duplicated rows: 0
Duplicate ID values in ['transaction_id']: 0

Outlier Analysis:


,Column,Valid Values,Minimum,Q1,Q3,Maximum,Lower Bound,Upper Bound,Outlier Count,Outlier Percentage (%)
0,amount_paise,110000,820.00,58255.00,271758.500,7.995933e+08,-262000.2500,592013.7500,10344,9.40
1,transaction_amount,110000,8.20,582.55,2717.585,7.995933e+06,-2620.0025,5920.1375,10344,9.40
2,customer_age,110000,18.00,34.00,64.000,8.000000e+01,-11.0000,109.0000,0,0.00
3,account_age,110000,0.00,468.00,1030.000,2.242000e+03,-375.0000,1873.0000,5601,5.09
4,previous_transactions,110000,0.00,1.00,6.000,3.500000e+01,-6.5000,13.5000,2530,2.30
5,average_transaction_value,107171,22.05,904.07,3096.930,3.998702e+06,-2385.2200,6386.2200,8652,8.07
6,failed_transactions,110000,0.00,0.00,0.000,8.000000e+00,0.0000,0.0000,16761,15.24



Split Analysis:


,Split,Row Count
0,train,72050
1,validation,14061
2,holdout,13889
3,warmup,10000


All four expected splits are present.
Total rows counted: 110,000


## Data quality report of payment_outcomes table

In [19]:
# Define expected data types for the payment_outcome table.
payment_outcomes_expected_types = {
    "transaction_id": "str",
    "outcome_time": "timedelta64[ns]",
    "status": "str",
    "failure_code": "str",
    "is_synthetic": "bool"
}

# Run all data quality checks on the complete payment_outcome dataset.
device_quality = run_data_quality_checks(
    df=sentinel_data["payment_outcomes"],
    table_name="payment_outcomes",
    expected_types=payment_outcomes_expected_types,
    id_columns=["transaction_id"],
    outlier_columns=[]
)


Data Quality Report: payment_outcomes
Total rows: 110,000
Total columns: 5

Column names:
['transaction_id', 'outcome_time', 'status', 'failure_code', 'is_synthetic']

Column data types:


,Data Type
transaction_id,str
outcome_time,str
status,str
failure_code,str
is_synthetic,bool


Data Type Validation:


,Column,Actual Type,Expected Type,Status
0,transaction_id,str,str,OK
1,outcome_time,str,timedelta64[ns],Review
2,status,str,str,OK
3,failure_code,str,str,OK
4,is_synthetic,bool,bool,OK



Missing Value Report:


,Null Count,Null Percentage (%)
transaction_id,0,0.00
outcome_time,0,0.00
status,0,0.00
failure_code,97985,89.08
is_synthetic,0,0.00



Duplicate Analysis:
Completely duplicated rows: 0
Duplicate ID values in ['transaction_id']: 0

Outlier Analysis:
Outlier analysis skipped: no suitable numeric columns.

Split Analysis:
The 'payment_outcomes' table does not contain a 'split' column. Split analysis is not applicable.


## Data quality report of labels table

In [ ]:
# Define expected data types for the labels table
labels_expected_types = {
    "transaction_id": "str",
    "fraud_label": "int64",
    "label_source": "str",
    "label_available_at": "timedelta64[ns]",
    "split": "str",
    "eligible_at_next_split_start": "bool",
    "is_synthetic": "bool"
}

# Run all data quality checks on the complete labels dataset
device_quality = run_data_quality_checks(
    df=sentinel_data["labels"],
    table_name="labels",
    expected_types=labels_expected_types,
    id_columns=["transaction_id"],
    outlier_columns=[]
)


Data Quality Report: labels
Total rows: 110,000
Total columns: 7

Column names:
['transaction_id', 'fraud_label', 'label_source', 'label_available_at', 'split', 'eligible_at_next_split_start', 'is_synthetic']

Column data types:


,Data Type
transaction_id,str
fraud_label,int64
label_source,str
label_available_at,str
split,str
eligible_at_next_split_start,bool
is_synthetic,bool


Data Type Validation:


,Column,Actual Type,Expected Type,Status
0,transaction_id,str,str,OK
1,fraud_label,int64,int64,OK
2,label_source,str,str,OK
3,label_available_at,str,timedelta64[ns],Review
4,split,str,str,OK
5,eligible_at_next_split_start,bool,bool,OK
6,is_synthetic,bool,bool,OK



Missing Value Report:


,Null Count,Null Percentage (%)
transaction_id,0,0.0
fraud_label,0,0.0
label_source,0,0.0
label_available_at,0,0.0
split,0,0.0
eligible_at_next_split_start,0,0.0
is_synthetic,0,0.0



Duplicate Analysis:
Completely duplicated rows: 0
Duplicate ID values in ['transaction_id']: 0

Outlier Analysis:
Outlier analysis skipped: no suitable numeric columns.

Split Analysis:


,Split,Row Count
0,train,72050
1,validation,14061
2,holdout,13889
3,warmup,10000


All four expected splits are present.
Total rows counted: 110,000


## Feature Engineering

Compute :

1. amount_to_average_ratio (amount divided by the customer's 90-day average) 
2. is_new_device (device not seen for this customer in 90 days)
3. customer_merchant_distance_km (haversine on latitude and longitude)
4. is_night
5. transaction_count_last_1_hour
6. transaction_amount_last_24_hours. 


Calculated 6 derived features using transaction history, customer, device details, location coordinates and transaction timestamps. All calculations use only past data by sorting transactions chronologically and excluding the current and future transactions to prevent data leakage.

In [21]:
# Access the transactions dataset from the sentinel_data dictionary
transactions_df = sentinel_data["transactions"].copy()

# Convert transaction_time from to datetime.
transactions_df["transaction_time"] = pd.to_datetime(transactions_df["transaction_time"], format="mixed", errors="coerce", utc=True)

# Sort transactions chronologically by customer
transactions_df = transactions_df.sort_values(["customer_id", "transaction_time"], kind="mergesort").reset_index(drop=True)

# Display dataset shape and updated data types.
print("\nTransactions dataset shape:", transactions_df.shape)
print("\nUpdated data types:")
display(transactions_df.dtypes)



Transactions dataset shape: (110000, 18)

Updated data types:


transaction_id                               str
customer_id                                  str
merchant_id                                  str
device_id                                    str
amount_paise                               int64
transaction_amount                       float64
payment_channel                              str
merchant_category                            str
transaction_time             datetime64[us, UTC]
customer_age                               int64
customer_location                         object
merchant_location                         object
account_age                                int64
previous_transactions                      int64
average_transaction_value                float64
failed_transactions                        int64
split                                        str
is_synthetic                                bool
dtype: object

In [ ]:
# FE-01: amount_to_average_ratio >> Current amount / customer's previous 90-day average

# Create a separate DataFrame using transaction IDs
derived_features_df = transactions_df[["transaction_id"]].copy()

# Initialize the new feature column
derived_features_df["amount_to_avg_ratio_derived"] = np.nan

# Calculate the previous 90-day average for each customer
for customer_id, group in transactions_df.groupby("customer_id", sort=False):

    # Set transaction time as the rolling window index
    time_indexed = group.set_index("transaction_time")

    # Calculate the average using only previous 90-day transactions
    previous_90d_avg = (time_indexed["transaction_amount"].rolling("90D", closed="left").mean())

    # Avoid division by zero.
    previous_90d_avg = previous_90d_avg.replace(0, np.nan)

    # Divide current transaction amount by its previous 90-day average
    ratio = (group["transaction_amount"].to_numpy()/ previous_90d_avg.to_numpy())

    # Store the feature in the separate DataFrame
    derived_features_df.loc[group.index, "amount_to_avg_ratio_derived"] = ratio

# Display the newly created feature table.
display(derived_features_df.head(10))

,transaction_id,amount_to_avg_ratio_derived
0,TX_0000000965,NaN
1,TX_0000000966,1.173731
2,TX_0000007393,3.739483
3,TX_0000007394,6.754269
4,TX_0000007395,0.801239
5,TX_0000008964,2.448720
6,TX_0000010732,0.249916
7,TX_0000012103,0.112308
8,TX_0000012104,0.460084
9,TX_0000014866,0.402535


In [ ]:
# FE-02: is_new_device >> Device not used by the customer in the previous 90 days

# Initialize the new-device feature in the separate feature DataFrame
derived_features_df["is_new_device_derived"] = pd.Series(pd.NA, index=transactions_df.index, dtype="boolean")

# Group transactions by customer and device
device_groups = transactions_df.dropna(subset=["device_id"]).groupby(["customer_id", "device_id"], sort=False)

# Check previous device usage for each customer-device combination
for _, group in device_groups:

    # Sort transactions chronologically.
    group = group.sort_values("transaction_time")

    # Track the most recent strictly earlier device usage.
    last_usage_time = None

    # Process transactions sharing the same timestamp together.
    for transaction_time, same_time_group in group.groupby("transaction_time", sort=True):

        # Calculate the beginning of the previous 90-day window.
        cutoff_time = transaction_time - pd.Timedelta(days=90)

        # Check whether the device was used within the previous 90 days.
        is_new = (last_usage_time is None or last_usage_time < cutoff_time)

        # Store the result in the separate feature DataFrame.
        derived_features_df.loc[same_time_group.index, "is_new_device_derived"] = is_new

        # Update the most recent device usage time.
        last_usage_time = transaction_time

# Display the newly created feature table.
display(derived_features_df.head(10))

,transaction_id,amount_to_avg_ratio_derived,is_new_device_derived
0,TX_0000000965,NaN,True
1,TX_0000000966,1.173731,False
2,TX_0000007393,3.739483,False
3,TX_0000007394,6.754269,False
4,TX_0000007395,0.801239,False
5,TX_0000008964,2.448720,False
6,TX_0000010732,0.249916,False
7,TX_0000012103,0.112308,False
8,TX_0000012104,0.460084,False
9,TX_0000014866,0.402535,False


In [ ]:
# Calculate geographical distance between two locations in kilometres
def haversine(lat1, lon1, lat2, lon2):

    # Earth's radius in kilometres
    earth_radius = 6371

    # Convert latitude and longitude from degrees to radians.
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Calculate differences between coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Apply the Haversine formula
    a = (math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2)

    # Calculate the angular distance
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Return the distance in kilometres
    return earth_radius * c

In [ ]:
# FE-03: customer_merchant_distance_km

# Calculate geographical distance using Haversine formula
def get_coordinates(location):

    # Handle missing location values.
    if location is None:
        return None, None

    # Convert JSON strings into Python dictionaries if necessary.
    if isinstance(location, str):
        try:
            location = json.loads(location)
        except (json.JSONDecodeError, TypeError):
            return None, None

    # Extract latitude and longitude from dictionaries.
    if isinstance(location, dict):
        lat = location.get("lat")
        lon = location.get("lon")

        try:
            if lat is not None and lon is not None:
                lat = float(lat)
                lon = float(lon)

                # Validate latitude and longitude ranges.
                if -90 <= lat <= 90 and -180 <= lon <= 180:
                    return lat, lon

        except (ValueError, TypeError):
            pass

    return None, None

# Initialize the geographical distance feature.
derived_features_df["customer_merchant_distance_km_derived"] = np.nan

# Calculate distance for each transaction.
for idx, row in transactions_df.iterrows():

    # Extract customer coordinates.
    customer_lat, customer_lon = get_coordinates(
        row["customer_location"]
    )

    # Extract merchant coordinates.
    merchant_lat, merchant_lon = get_coordinates(
        row["merchant_location"]
    )

    # Calculate distance if both locations are valid.
    if customer_lat is not None and merchant_lat is not None:

        # Calculate distance using the Haversine formula.
        distance = haversine(
            customer_lat, customer_lon,
            merchant_lat, merchant_lon
        )


        # Store the distance in the separate feature DataFrame.
        derived_features_df.loc[
            idx, "customer_merchant_distance_km_derived"
        ] = distance

# Display the newly created feature table.
display(derived_features_df.head(10))

,transaction_id,amount_to_avg_ratio_derived,is_new_device_derived,customer_merchant_distance_km_derived
0,TX_0000000965,NaN,True,1144.866173
1,TX_0000000966,1.173731,False,NaN
2,TX_0000007393,3.739483,False,920.081741
3,TX_0000007394,6.754269,False,920.081741
4,TX_0000007395,0.801239,False,920.081741
5,TX_0000008964,2.448720,False,1257.929214
6,TX_0000010732,0.249916,False,953.640826
7,TX_0000012103,0.112308,False,1888.762166
8,TX_0000012104,0.460084,False,1888.762166
9,TX_0000014866,0.402535,False,1094.365822


In [ ]:
# FE-04: is_night >> Night-time means midnight to 4:59 AM

# Convert UTC timestamps into Indian Standard Time.
local_transaction_time = transactions_df["transaction_time"].dt.tz_convert("Asia/Kolkata")

# Identify transactions between midnight and 4:59 AM.
is_night = ((local_transaction_time.dt.hour >= 0)  & (local_transaction_time.dt.hour < 5))

# Store the feature in the separate feature DataFrame.
derived_features_df["is_night_derived"] = is_night.astype("boolean")

# Display the newly created feature table.
display(derived_features_df.head(10))

,transaction_id,amount_to_avg_ratio_derived,is_new_device_derived,customer_merchant_distance_km_derived,is_night_derived
0,TX_0000000965,NaN,True,1144.866173,False
1,TX_0000000966,1.173731,False,NaN,False
2,TX_0000007393,3.739483,False,920.081741,False
3,TX_0000007394,6.754269,False,920.081741,False
4,TX_0000007395,0.801239,False,920.081741,False
5,TX_0000008964,2.448720,False,1257.929214,False
6,TX_0000010732,0.249916,False,953.640826,False
7,TX_0000012103,0.112308,False,1888.762166,False
8,TX_0000012104,0.460084,False,1888.762166,False
9,TX_0000014866,0.402535,False,1094.365822,False


In [ ]:
# FE-07: transaction_count_last_1_hour

# Initialize the feature in the separate feature DataFrame.
derived_features_df["transaction_count_last_1_hour_derived"] = np.nan

# Calculate transaction counts separately for each customer.
for customer_id, group in transactions_df.groupby("customer_id", sort=False):

    # Keep only transactions with valid timestamps and sort chronologically.
    valid_group = group.dropna(subset=["transaction_time"]).sort_values("transaction_time")

    # Set transaction time as the rolling window index.
    time_indexed = valid_group.set_index("transaction_time")

    # Count transactions within the previous one hour.
    previous_1h_count = time_indexed["transaction_id"].rolling("1h", closed="left").count()   
    
    # Replace missing rolling counts with zero
    previous_1h_count = previous_1h_count.fillna(0)

    # Store the calculated counts in the separate feature DataFrame.
    derived_features_df.loc[valid_group.index, "transaction_count_last_1_hour_derived"] = previous_1h_count.to_numpy()

    # Ensure the complete column uses integer storage.
    derived_features_df["transaction_count_last_1_hour_derived"] = (derived_features_df["transaction_count_last_1_hour_derived"])

# Display the newly created feature table.
display(derived_features_df.head(10))

,transaction_id,amount_to_avg_ratio_derived,is_new_device_derived,customer_merchant_distance_km_derived,is_night_derived,transaction_count_last_1_hour_derived
0,TX_0000000965,NaN,True,1144.866173,False,0.0
1,TX_0000000966,1.173731,False,NaN,False,1.0
2,TX_0000007393,3.739483,False,920.081741,False,0.0
3,TX_0000007394,6.754269,False,920.081741,False,1.0
4,TX_0000007395,0.801239,False,920.081741,False,2.0
5,TX_0000008964,2.448720,False,1257.929214,False,0.0
6,TX_0000010732,0.249916,False,953.640826,False,0.0
7,TX_0000012103,0.112308,False,1888.762166,False,0.0
8,TX_0000012104,0.460084,False,1888.762166,False,1.0
9,TX_0000014866,0.402535,False,1094.365822,False,0.0


In [ ]:
# FE-08: transaction_amount_last_24_hours

# Initialize the feature in the separate feature DataFrame
derived_features_df["transaction_amount_last_24_hours_derived"] = np.nan

# Calculate transaction amounts separately for each customer
for customer_id, group in transactions_df.groupby("customer_id", sort=False):

    # Keep valid timestamps and sort transactions chronologically
    valid_group = group.dropna(subset=["transaction_time"]).sort_values("transaction_time")

    # Set transaction time as the rolling window index.
    time_indexed = valid_group.set_index("transaction_time")

    # Calculate the previous 24-hour transaction amount.
    previous_24h_amount = time_indexed["transaction_amount"].rolling("24h", closed="left").sum()

    # Replace missing rolling sums with zero.
    previous_24h_amount = previous_24h_amount.fillna(0)

    # Store the calculated amounts in the separate feature DataFrame
    derived_features_df.loc[valid_group.index, "transaction_amount_last_24_hours_derived"] = previous_24h_amount.to_numpy()

# Display the newly created feature table.
display(derived_features_df.head(10))

,transaction_id,amount_to_avg_ratio_derived,is_new_device_derived,customer_merchant_distance_km_derived,is_night_derived,transaction_count_last_1_hour_derived,transaction_amount_last_24_hours_derived
0,TX_0000000965,NaN,True,1144.866173,False,0.0,0.00
1,TX_0000000966,1.173731,False,NaN,False,1.0,3647.83
2,TX_0000007393,3.739483,False,920.081741,False,0.0,0.00
3,TX_0000007394,6.754269,False,920.081741,False,1.0,14825.93
4,TX_0000007395,0.801239,False,920.081741,False,2.0,66057.80
5,TX_0000008964,2.448720,False,1257.929214,False,0.0,0.00
6,TX_0000010732,0.249916,False,953.640826,False,0.0,0.00
7,TX_0000012103,0.112308,False,1888.762166,False,0.0,0.00
8,TX_0000012104,0.460084,False,1888.762166,False,1.0,2917.36
9,TX_0000014866,0.402535,False,1094.365822,False,0.0,0.00


In [ ]:
# 11. Save derived features locally

# Define the local output folder.
output_folder = Path("data")

# Create the folder if it does not already exist.
output_folder.mkdir(parents=True, exist_ok=True)

# Define the output CSV file path.
output_path = output_folder / "derived_features.csv"

# Save the engineered features locally.
derived_features_df.to_csv(output_path, index=False)

# Display final file information.
print("\nFeature engineering completed successfully!")
print("Total records:", len(derived_features_df))
print("Total columns:", len(derived_features_df.columns))


Feature engineering completed successfully!
Total records: 110000
Total columns: 7


In [116]:
# Define SENTINEL reason-code descriptions.
REASON_CODES = {
    "R01": "NEW_DEVICE",
    "R02": "GEO_MISMATCH",
    "R03": "ODD_HOUR",
    "R04": "FAILED_ATTEMPTS",
    "R05": "AMOUNT_SPIKE",
    "R06": "NEW_ACCOUNT",
    "R07": "VELOCITY",
    "R08": "HIGH_RISK_MCC"
}

In [ ]:
# Store temporary SENTINEL rule thresholds

RULE_THRESHOLDS = {
    "R01": {
        "name": "NEW_DEVICE",
        "device_history_days": 90
    },
    "R02": {
        "name": "GEO_MISMATCH",
        "distance_km": 500
    },
    "R03": {
        "name": "ODD_HOUR",
        "start_hour": 0,
        "end_hour": 5
    },
    "R04": {
        "name": "FAILED_ATTEMPTS",
        "min_failed_attempts": 3,
        "window_hours": 24
    },
    "R05": {
        "name": "AMOUNT_SPIKE",
        "amount_ratio": 5
    },
    "R06": {
        "name": "NEW_ACCOUNT",
        "account_age_days": 7
    },
    "R07": {
        "name": "VELOCITY",
        "max_transactions": 5,
        "window_minutes": 60
    },
    "R08": {
        "name": "HIGH_RISK_MCC",
        "high_risk_mcc_codes": ["travel",
        "gaming",
        "electronics"]
    }
}

In [ ]:
# Display threshold configuration.
for code, details in RULE_THRESHOLDS.items():
    print(code, details)

R01 {'name': 'NEW_DEVICE', 'device_history_days': 90}
R02 {'name': 'GEO_MISMATCH', 'distance_km': 500}
R03 {'name': 'ODD_HOUR', 'start_hour': 0, 'end_hour': 5}
R04 {'name': 'FAILED_ATTEMPTS', 'min_failed_attempts': 3, 'window_hours': 24}
R05 {'name': 'AMOUNT_SPIKE', 'amount_ratio': 5}
R06 {'name': 'NEW_ACCOUNT', 'account_age_days': 7}
R07 {'name': 'VELOCITY', 'max_transactions': 5, 'window_minutes': 60}
R08 {'name': 'HIGH_RISK_MCC', 'high_risk_mcc_codes': ['travel', 'gaming', 'electronics']}


In [119]:
print("Features:")
print(derived_features_df.columns.tolist())

print("\nThreshold configuration:")
print(REASON_CODES)

Features:
['transaction_id', 'amount_to_avg_ratio_derived', 'is_new_device_derived', 'customer_merchant_distance_km_derived', 'is_night_derived', 'transaction_count_last_1_hour_derived', 'transaction_amount_last_24_hours_derived']

Threshold configuration:
{'R01': 'NEW_DEVICE', 'R02': 'GEO_MISMATCH', 'R03': 'ODD_HOUR', 'R04': 'FAILED_ATTEMPTS', 'R05': 'AMOUNT_SPIKE', 'R06': 'NEW_ACCOUNT', 'R07': 'VELOCITY', 'R08': 'HIGH_RISK_MCC'}


In [120]:
derived_features_df.isnull().sum()

transaction_id                                 0
amount_to_avg_ratio_derived                 2829
is_new_device_derived                       1138
customer_merchant_distance_km_derived       3257
is_night_derived                               0
transaction_count_last_1_hour_derived          0
transaction_amount_last_24_hours_derived       0
dtype: int64

In [ ]:
# Create a separate DataFrame 
reason_codes_df = transactions_df[["transaction_id"]].copy()

# Initialize all reason codes to zero.
for code in REASON_CODES:
    reason_codes_df[code] = 0
display(reason_codes_df.head())

,transaction_id,R01,R02,R03,R04,R05,R06,R07,R08
0,TX_0000000965,0,0,0,0,0,0,0,0
1,TX_0000000966,0,0,0,0,0,0,0,0
2,TX_0000007393,0,0,0,0,0,0,0,0
3,TX_0000007394,0,0,0,0,0,0,0,0
4,TX_0000007395,0,0,0,0,0,0,0,0


In [122]:
# R01: New device
reason_codes_df["R01"] = (
    derived_features_df["is_new_device_derived"]
    .eq(1)
    .fillna(False)
    .astype("int8")
)

# R02: Geographical mismatch
reason_codes_df["R02"] = (
    derived_features_df["customer_merchant_distance_km_derived"]
    >= RULE_THRESHOLDS["R02"]["distance_km"]
).fillna(False).astype("int8")

# R03: Odd-hour transaction
reason_codes_df["R03"] = (
    derived_features_df["is_night_derived"] == True
).fillna(False).astype("int8")

# R04: Multiple failed attempts
reason_codes_df["R04"] = (
    transactions_df["failed_transactions"]
    >= RULE_THRESHOLDS["R04"]["min_failed_attempts"]
).fillna(False).astype("int8")

# R05: Unusually high transaction amount
reason_codes_df["R05"] = (
    derived_features_df["amount_to_avg_ratio_derived"]
    >= RULE_THRESHOLDS["R05"]["amount_ratio"]
).fillna(False).astype("int8")

# R06: Recently created account
reason_codes_df["R06"] = (
    transactions_df["account_age"]
    <= RULE_THRESHOLDS["R06"]["account_age_days"]
).fillna(False).astype("int8")

In [123]:
reason_codes_df

,transaction_id,R01,R02,R03,R04,R05,R06,R07,R08
0,TX_0000000965,1,1,0,0,0,0,0,0
1,TX_0000000966,0,0,0,0,0,0,0,0
2,TX_0000007393,0,1,0,0,0,0,0,0
3,TX_0000007394,0,1,0,0,1,0,0,0
4,TX_0000007395,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
109995,TX_0000102457,0,0,0,0,0,0,0,0
109996,TX_0000104669,0,1,0,0,0,0,0,0
109997,TX_0000105698,0,0,1,0,0,0,0,0
109998,TX_0000106370,0,0,0,0,0,0,0,0


In [ ]:
# Display the transaction details for the given transaction ID.
transactions_df[transactions_df["transaction_id"] == "TX_0000000965"]

,transaction_id,customer_id,merchant_id,device_id,amount_paise,transaction_amount,payment_channel,merchant_category,transaction_time,customer_age,customer_location,merchant_location,account_age,previous_transactions,average_transaction_value,failed_transactions,split,is_synthetic
0,TX_0000000965,C_0004ff2f7739f0d3,M_bdfbd509bd290325,D_506a929cba09f1ad,364783,3647.83,card,travel,2024-10-09 13:38:16.727+00:00,60,"{'lat': 26.91, 'lon': 75.79, 'city': 'Jaipur'}","{'lat': 22.8, 'lon': 86.2, 'city': 'Jamshedpur'}",555,0,NaN,0,warmup,True


In [ ]:
# Display the engineered features for the same transaction.
derived_features_df[derived_features_df["transaction_id"] == "TX_0000000965"]

,transaction_id,amount_to_avg_ratio_derived,is_new_device_derived,customer_merchant_distance_km_derived,is_night_derived,transaction_count_last_1_hour_derived,transaction_amount_last_24_hours_derived
0,TX_0000000965,NaN,True,1144.866173,False,0.0,0.0


In [ ]:
reason_codes_df[reason_codes_df["transaction_id"] == "TX_0000000965"]

,transaction_id,R01,R02,R03,R04,R05,R06,R07,R08
0,TX_0000000965,1,1,0,0,0,0,0,0


In [127]:
# Define expected data types for the derived_features table.
derived_features_expected_types = {
    "transaction_id": "str",
    "amount_to_avg_ratio_derived": "float64",
    "is_new_device_derived": "bool",
    "customer_merchant_distance_km_derived": "float64",
    "is_night_derived": "bool",
    "transaction_count_last_1_hour_derived": "float",
    "transaction_amount_last_24_hours_derived": "float64"
}

# Run all data quality checks on the complete derived_features dataset.
device_quality = run_data_quality_checks(
    df=derived_features_df,
    table_name="derived_features",
    expected_types=derived_features_expected_types,
    id_columns=["transaction_id"],
    outlier_columns=[]
)


Data Quality Report: derived_features
Total rows: 110,000
Total columns: 7

Column names:
['transaction_id', 'amount_to_avg_ratio_derived', 'is_new_device_derived', 'customer_merchant_distance_km_derived', 'is_night_derived', 'transaction_count_last_1_hour_derived', 'transaction_amount_last_24_hours_derived']

Column data types:


,Data Type
transaction_id,str
amount_to_avg_ratio_derived,float64
is_new_device_derived,boolean
customer_merchant_distance_km_derived,float64
is_night_derived,boolean
transaction_count_last_1_hour_derived,float64
transaction_amount_last_24_hours_derived,float64


Data Type Validation:


,Column,Actual Type,Expected Type,Status
0,transaction_id,str,str,OK
1,amount_to_avg_ratio_derived,float64,float64,OK
2,is_new_device_derived,boolean,bool,Review
3,customer_merchant_distance_km_derived,float64,float64,OK
4,is_night_derived,boolean,bool,Review
5,transaction_count_last_1_hour_derived,float64,float,Review
6,transaction_amount_last_24_hours_derived,float64,float64,OK



Missing Value Report:


,Null Count,Null Percentage (%)
transaction_id,0,0.00
amount_to_avg_ratio_derived,2829,2.57
is_new_device_derived,1138,1.03
customer_merchant_distance_km_derived,3257,2.96
is_night_derived,0,0.00
transaction_count_last_1_hour_derived,0,0.00
transaction_amount_last_24_hours_derived,0,0.00



Duplicate Analysis:
Completely duplicated rows: 0
Duplicate ID values in ['transaction_id']: 0

Outlier Analysis:
Outlier analysis skipped: no suitable numeric columns.

Split Analysis:
The 'derived_features' table does not contain a 'split' column. Split analysis is not applicable.


In [ ]:
# Get unique local transaction IDs
local_transaction_ids = (derived_features_df["transaction_id"].dropna().drop_duplicates()
)

# Select a reproducible 10% sample
sampled_transaction_ids = local_transaction_ids.sample(frac=0.10, random_state=42).tolist()

print("Total local transaction IDs:", len(local_transaction_ids))
print("Selected transaction IDs:", len(sampled_transaction_ids))

Total local transaction IDs: 110000
Selected transaction IDs: 11000


In [ ]:
def load_features_by_ids(transaction_ids, batch_size=200):

    # Store fetched records
    all_records = []

    # Fetch transaction IDs in smaller batches
    for start in range(0, len(transaction_ids), batch_size):

        batch_ids = transaction_ids[start:start + batch_size]
        response = (supabase.table("features").select("*").in_("transaction_id", batch_ids).execute())
        all_records.extend(response.data)

        print(
            f"Fetched {min(start + batch_size, len(transaction_ids))}"
            f" / {len(transaction_ids)} transaction IDs"
        )

    return pd.DataFrame(all_records)

In [ ]:
# Fetch only the selected transaction IDs.
features_sample_df = load_features_by_ids(sampled_transaction_ids, batch_size=200)

print("\nSupabase sample shape:", features_sample_df.shape)
display(features_sample_df.head())

Fetched 200 / 11000 transaction IDs
Fetched 400 / 11000 transaction IDs
Fetched 600 / 11000 transaction IDs
Fetched 800 / 11000 transaction IDs
Fetched 1000 / 11000 transaction IDs
Fetched 1200 / 11000 transaction IDs
Fetched 1400 / 11000 transaction IDs
Fetched 1600 / 11000 transaction IDs
Fetched 1800 / 11000 transaction IDs
Fetched 2000 / 11000 transaction IDs
Fetched 2200 / 11000 transaction IDs
Fetched 2400 / 11000 transaction IDs
Fetched 2600 / 11000 transaction IDs
Fetched 2800 / 11000 transaction IDs
Fetched 3000 / 11000 transaction IDs
Fetched 3200 / 11000 transaction IDs
Fetched 3400 / 11000 transaction IDs
Fetched 3600 / 11000 transaction IDs
Fetched 3800 / 11000 transaction IDs
Fetched 4000 / 11000 transaction IDs
Fetched 4200 / 11000 transaction IDs
Fetched 4400 / 11000 transaction IDs
Fetched 4600 / 11000 transaction IDs
Fetched 4800 / 11000 transaction IDs
Fetched 5000 / 11000 transaction IDs
Fetched 5200 / 11000 transaction IDs
Fetched 5400 / 11000 transaction IDs
Fetch

,transaction_id,feature_as_of,amount_to_avg_ratio,is_new_device,customer_merchant_distance_km,is_new_city,hour_of_day,is_night,has_recent_failures,is_new_account,...,distinct_customers_per_device_7d,combined_risk_flags,mcc_target_encoded_rate,mcc_encoding_as_of,mcc_encoding_count,missing_customer_location,missing_device,cold_start,reason_codes,feature_version
0,TX_0000000138,2024-10-02T04:45:36.414+00:00,NaN,True,367.36,True,10,False,False,False,...,0.0,2,0.006,2024-09-30T18:30:00+00:00,0,False,False,True,"[R01_NEW_DEVICE, R02_GEO_MISMATCH, R08_HIGH_RI...",sentinel-features-1.0.0
1,TX_0000000207,2024-10-02T15:04:44.055+00:00,NaN,True,594.53,True,20,False,False,False,...,0.0,2,0.006,2024-09-30T18:30:00+00:00,0,False,False,True,"[R01_NEW_DEVICE, R02_GEO_MISMATCH]",sentinel-features-1.0.0
2,TX_0000002493,2024-10-23T21:46:15.158+00:00,0.6459,False,524.16,True,3,True,False,False,...,0.0,2,0.006,2024-09-30T18:30:00+00:00,0,False,False,False,"[R02_GEO_MISMATCH, R03_ODD_HOUR, R08_HIGH_RISK...",sentinel-features-1.0.0
3,TX_0000002823,2024-10-26T16:56:32.163+00:00,NaN,True,1254.43,True,22,False,False,False,...,0.0,2,0.006,2024-09-30T18:30:00+00:00,0,False,False,True,"[R01_NEW_DEVICE, R02_GEO_MISMATCH]",sentinel-features-1.0.0
4,TX_0000003473,2024-11-01T15:06:57.988+00:00,0.2459,False,1196.54,False,20,False,False,False,...,1.0,1,0.006,2024-10-31T18:30:00+00:00,0,False,False,False,[R02_GEO_MISMATCH],sentinel-features-1.0.0


In [ ]:
# Filter local features using the same transaction IDs.
derived_features_sample_df = derived_features_df[derived_features_df["transaction_id"].isin(sampled_transaction_ids)].copy()
print("Local sample shape:", derived_features_sample_df.shape)

Local sample shape: (11000, 7)


In [132]:
# Compare transaction IDs between both samples.
supabase_ids = set(features_sample_df["transaction_id"])
local_ids = set(derived_features_sample_df["transaction_id"])

print("Matching transaction IDs:", len(supabase_ids & local_ids))
print("Missing from Supabase:", len(local_ids - supabase_ids))
print("Missing from local:", len(supabase_ids - local_ids))

Matching transaction IDs: 11000
Missing from Supabase: 0
Missing from local: 0


In [133]:
# Display all Supabase feature columns.
print("Supabase columns:")
for column in features_sample_df.columns:
    print(column, ":", features_sample_df[column].dtype)

# Display local feature columns.
print("\nLocal derived feature columns:")
for column in derived_features_sample_df.columns:
    print(column, ":", derived_features_sample_df[column].dtype)

Supabase columns:
transaction_id : str
feature_as_of : str
amount_to_avg_ratio : float64
is_new_device : object
customer_merchant_distance_km : float64
is_new_city : object
hour_of_day : int64
is_night : bool
has_recent_failures : bool
is_new_account : bool
txn_count_last_1h : int64
txn_amount_last_24h_paise : int64
distinct_customers_per_device_7d : float64
combined_risk_flags : int64
mcc_target_encoded_rate : float64
mcc_encoding_as_of : str
mcc_encoding_count : int64
missing_customer_location : bool
missing_device : bool
cold_start : bool
reason_codes : object
feature_version : str

Local derived feature columns:
transaction_id : str
amount_to_avg_ratio_derived : float64
is_new_device_derived : boolean
customer_merchant_distance_km_derived : float64
is_night_derived : boolean
transaction_count_last_1_hour_derived : float64
transaction_amount_last_24_hours_derived : float64


In [ ]:
derived_features_sample_df = derived_features_df[derived_features_df["transaction_id"].isin(sampled_transaction_ids)].copy()

In [ ]:
# Map local feature names to Supabase feature names
feature_mapping = {
    "amount_to_avg_ratio": "amount_to_avg_ratio_derived",
    "is_new_device": "is_new_device_derived",
    "customer_merchant_distance_km": "customer_merchant_distance_km_derived",
    "is_night": "is_night_derived",
    "txn_count_last_1h": "transaction_count_last_1_hour_derived",
    "txn_amount_last_24h_paise": "transaction_amount_last_24_hours_derived"
}

# Check for duplicate transaction IDs
print("Supabase duplicate IDs:",
      features_sample_df["transaction_id"].duplicated().sum())
print("Local duplicate IDs:",
      derived_features_sample_df["transaction_id"].duplicated().sum())

Supabase duplicate IDs: 0
Local duplicate IDs: 0


In [ ]:
derived_features_sample_df["transaction_amount_last_24_hours_derived"] *= 100

In [156]:
comparison_df = features_sample_df.merge(
    derived_features_sample_df,
    on="transaction_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_supabase", "_local")
)

print("Merged records:", len(comparison_df))

Merged records: 11000


In [157]:
comparison_df.head()

,transaction_id,feature_as_of,amount_to_avg_ratio,is_new_device,customer_merchant_distance_km,is_new_city,hour_of_day,is_night,has_recent_failures,is_new_account,...,missing_device,cold_start,reason_codes,feature_version,amount_to_avg_ratio_derived,is_new_device_derived,customer_merchant_distance_km_derived,is_night_derived,transaction_count_last_1_hour_derived,transaction_amount_last_24_hours_derived
0,TX_0000000138,2024-10-02T04:45:36.414+00:00,NaN,True,367.36,True,10,False,False,False,...,False,True,"[R01_NEW_DEVICE, R02_GEO_MISMATCH, R08_HIGH_RI...",sentinel-features-1.0.0,NaN,True,367.362405,False,0.0,0.0
1,TX_0000000207,2024-10-02T15:04:44.055+00:00,NaN,True,594.53,True,20,False,False,False,...,False,True,"[R01_NEW_DEVICE, R02_GEO_MISMATCH]",sentinel-features-1.0.0,NaN,True,594.531453,False,0.0,0.0
2,TX_0000002493,2024-10-23T21:46:15.158+00:00,0.6459,False,524.16,True,3,True,False,False,...,False,False,"[R02_GEO_MISMATCH, R03_ODD_HOUR, R08_HIGH_RISK...",sentinel-features-1.0.0,0.645935,False,524.163382,True,0.0,0.0
3,TX_0000002823,2024-10-26T16:56:32.163+00:00,NaN,True,1254.43,True,22,False,False,False,...,False,True,"[R01_NEW_DEVICE, R02_GEO_MISMATCH]",sentinel-features-1.0.0,NaN,True,1254.433580,False,0.0,0.0
4,TX_0000003473,2024-11-01T15:06:57.988+00:00,0.2459,False,1196.54,False,20,False,False,False,...,False,False,[R02_GEO_MISMATCH],sentinel-features-1.0.0,0.245923,False,1196.544771,False,1.0,6101900.0


In [ ]:
comparison_results = []
mismatch_details = {}

for supabase_col, local_col in feature_mapping.items():

    supabase_values = comparison_df[supabase_col]
    local_values = comparison_df[local_col]

    # Identify missing values on each side.
    both_missing = (supabase_values.isna() & local_values.isna())

    one_missing = (supabase_values.isna() ^ local_values.isna())

    valid = (supabase_values.notna() & local_values.notna())

    # Numerical feature comparison.
    if supabase_col in [
        "amount_to_avg_ratio",
        "customer_merchant_distance_km",
        "txn_count_last_1h",
        "txn_amount_last_24h_paise"
    ]:

        matches = pd.Series(False, index=comparison_df.index)

        matches.loc[valid] = np.isclose(
            pd.to_numeric(local_values[valid]),
            pd.to_numeric(supabase_values[valid]),
            rtol=1e-5,
            atol=1e-6
        )

    # Boolean feature comparison.
    else:

        matches = (
            local_values.astype("string").str.lower()
            == supabase_values.astype("string").str.lower()
        )

    match_count = int((matches & valid).sum())
    mismatch_count = int((~matches & valid).sum())

    comparable_count = match_count + mismatch_count

    match_rate = (
        match_count / comparable_count * 100
        if comparable_count else np.nan
    )

    comparison_results.append({
        "Feature": supabase_col,
        "Matching Values": match_count,
        "Mismatching Values": mismatch_count,
        "Both Missing": int(both_missing.sum()),
        "Only One Missing": int(one_missing.sum()),
        "Match Rate (%)": round(match_rate, 2)
    })

    # Store mismatched records.
    mismatch_details[supabase_col] = comparison_df.loc[
        valid & ~matches,
        ["transaction_id", supabase_col, local_col]
    ].copy()

comparison_summary_df = pd.DataFrame(comparison_results)

display(comparison_summary_df)

,Feature,Matching Values,Mismatching Values,Both Missing,Only One Missing,Match Rate (%)
0,amount_to_avg_ratio,2519,8218,263,0,23.46
1,is_new_device,10891,0,109,0,100.00
2,customer_merchant_distance_km,10505,164,331,0,98.46
3,is_night,11000,0,0,0,100.00
4,txn_count_last_1h,11000,0,0,0,100.00
5,txn_amount_last_24h_paise,5980,5020,0,0,54.36


Variations in match rates may be due to the differences in feature engineering approaches, historical data handling, calculation logic and implementation assumptions between the Supabase-generated features and locally derived features. Further investigation is required to identify the exact causes of individual mismatches.